In [6]:
import pandas as pd

THRESHOLD = 299.9

df = pd.read_csv('results.csv')

# Clean up times
time_cols = {
    'Proposed SAT': 'SAT TIME (s)',
    'CPLEX MP':     'MP TIME (s)',
    'CPLEX CP':     'CP TIME (s)',
    'Gurobi':       'GUROBI TIME (s)',
    'Basic SAT':    'BSAT TIME (s)',
}

for label, col in time_cols.items():
    df[col] = df[col].replace('TIMEOUT', 300)
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Extract parameters from filename
df['n']   = df['FILENAME'].str.split('_').str[0].astype(int)
df['tau'] = df['FILENAME'].str.split('_').str[1].astype(int).map({5: 0.5, 10: 1.0})
df['rho'] = df['FILENAME'].str.split('_').str[2].astype(int).map({5: 0.05, 25: 0.25, 50: 0.50})
df['phi'] = df['FILENAME'].str.split('_').str[3].astype(int).map({100: 1.00, 125: 1.25})
df['OS']  = df['FILENAME'].str.split('_').str[4].astype(int).map({25: 0.25, 50: 0.50, 75: 0.75})

def solve_rate_table(param, param_label, data):
    rows = []
    for val in sorted(data[param].unique()):
        subset = data[data[param] == val]
        row = {param_label: val}
        for label, col in time_cols.items():
            solved = (subset[col] < THRESHOLD).sum()
            total  = len(subset)
            row[label] = f'{solved}/{total} ({100*solved/total:.1f}%)'
        rows.append(row)
    table = pd.DataFrame(rows).set_index(param_label)
    return table

for param, label in [('TYPE', 'Processing Time'), ('tau', 'tau'), 
                     ('rho', 'rho'), ('phi', 'phi'), ('OS', 'OS')]:
    print(f'\n=== Solve Rate by {label} (n >= 30) ===')
    print(solve_rate_table(param, label, df).to_string())


=== Solve Rate by Processing Time (n >= 30) ===
                    Proposed SAT         CPLEX MP          CPLEX CP           Gurobi        Basic SAT
Processing Time                                                                                      
L                155/174 (89.1%)  121/174 (69.5%)  174/174 (100.0%)  124/174 (71.3%)  105/174 (60.3%)
S                168/180 (93.3%)  131/180 (72.8%)  180/180 (100.0%)  132/180 (73.3%)  159/180 (88.3%)

=== Solve Rate by tau (n >= 30) ===
         Proposed SAT         CPLEX MP          CPLEX CP           Gurobi        Basic SAT
tau                                                                                       
0.5   149/180 (82.8%)   92/180 (51.1%)  180/180 (100.0%)   95/180 (52.8%)  106/180 (58.9%)
1.0  174/174 (100.0%)  160/174 (92.0%)  174/174 (100.0%)  161/174 (92.5%)  158/174 (90.8%)

=== Solve Rate by rho (n >= 30) ===
         Proposed SAT        CPLEX MP          CPLEX CP          Gurobi       Basic SAT
rho              

In [7]:
df_hard = df[df['n'] >= 30]

for param, label in [('TYPE', 'Processing Time'), ('tau', 'tau'), 
                     ('rho', 'rho'), ('phi', 'phi'), ('OS', 'OS')]:
    print(f'\n=== Solve Rate by {label} (n >= 30) ===')
    print(solve_rate_table(param, label, df_hard).to_string())


=== Solve Rate by Processing Time (n >= 30) ===
                   Proposed SAT        CPLEX MP          CPLEX CP          Gurobi       Basic SAT
Processing Time                                                                                  
L                89/108 (82.4%)  56/108 (51.9%)  108/108 (100.0%)  58/108 (53.7%)  42/108 (38.9%)
S                96/108 (88.9%)  59/108 (54.6%)  108/108 (100.0%)  60/108 (55.6%)  87/108 (80.6%)

=== Solve Rate by tau (n >= 30) ===
         Proposed SAT        CPLEX MP          CPLEX CP          Gurobi       Basic SAT
tau                                                                                    
0.5    77/108 (71.3%)  21/108 (19.4%)  108/108 (100.0%)  23/108 (21.3%)  37/108 (34.3%)
1.0  108/108 (100.0%)  94/108 (87.0%)  108/108 (100.0%)  95/108 (88.0%)  92/108 (85.2%)

=== Solve Rate by rho (n >= 30) ===
       Proposed SAT       CPLEX MP        CPLEX CP         Gurobi      Basic SAT
rho                                                 